#### House Price Prediction Analysis


In [1]:
# install required libraries
!pip install psycopg2-binary sqlalchemy pandas


In [32]:
#connecting to PostgreSQl and read the table
import pandas as pd
from sqlalchemy import create_engine
from datetime import datetime

# defining the connection parameters
db_user = "postgres"
db_password = "Shutterbug1,"
db_host = "localhost"
db_port = "5432"
db_name = "postgres"

# creating the SQLAlchemy engine
engine = create_engine(f"postgresql+psycopg2://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}")

# reading the table into a Dataframe
df = pd.read_sql("SELECT * FROM africa.property_data", engine)

# preview the Dataframe
df.head(20)

,BedRooms,Bath rooms,Size_sqft,location,year built,Garage Available,Furnishing,House Condition,Has_Pool,Lot Size,Price($)
0,3.0,1.0,1149,Rural,2004,NaN,Unfurnished,new,0.0,0.15,244043
1,NaN,2.0,1169,Rural,1989,1.0,,Old,0.0,0.23,211250
2,4.0,3.0,1409,Suburban,1993,1.0,Unfurnished,gd,0.0,0.19,257239
3,3.0,3.0,1647,Suburban,2021,0.0,,new,0.0,0.27,310316
4,2.0,3.0,1865,Urban,2009,0.0,furnised,old,1.0,0.39,334791
5,2.0,2.0,1179,ruraal,2020,NaN,Semi furnished,Old,0.0,0.32,228260
6,1.0,2.0,2097,Suburban,2004,0.0,Semi-Furnished,new,0.0,0.33,361672
7,5.0,3.0,1392,Suburban,1993,1.0,Semi furnished,new,0.0,0.12,259003
8,3.0,1.0,1491,Rural,2014,0.0,Furnished,old,0.0,0.01,263096
9,4.0,2.0,1201,ruraal,2013,0.0,Semi-Furnished,gd,0.0,0.13,242759


#### explanation of data importation
- pandas is used for data manipulation and analysis
- create_engine from SQLAlchemy helps create a connection to the PostgreSQL database
- the database credidentials and connection details were then input
- engine = create_engine(f"postgresql+psycopg2://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}")
-   this line creates a connection engine using SQLAlchemy
    postgresql+psycopg2 specifies the database dialect and driver
- to breakdown in simple terms how I created the SQLAlchemy engine:“Use the postgres user and password Shutterbug1, to connect to a PostgreSQL server on localhost at port 5432, and access the postgres database.”
- Colon (:): Commonly used to separate values in key:value format (e.g., user:pass or host:port).
- At symbol (@): Separates authentication details from host info, just like in email (anangwemike@gmail.com).
- Slash (/): Used to indicate hierarchy or location, like file paths or URLs (e.g., which database inside the server).
- df = pd.read_sql("SELECT * FROM africa.property_data", engine)
-   the SQL query fetches all records from the property_data table inside the africa schema
-   the result is stored in a pandas Dataframe "df" that allows me to analyse and manipulate the data

#### Cleaning the dataframe

In [20]:
# 2)understand the data structure
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0    BedRooms         954 non-null    float64
 1   Bath rooms        941 non-null    float64
 2   Size_sqft         1000 non-null   int64  
 3   location          1000 non-null   object 
 4   year built        1000 non-null   int64  
 5   Garage Available  896 non-null    float64
 6   Furnishing        1000 non-null   object 
 7   House Condition   1000 non-null   object 
 8   Has_Pool          945 non-null    float64
 9   Lot Size          1000 non-null   float64
 10  Price($)          1000 non-null   int64  
dtypes: float64(5), int64(3), object(3)
memory usage: 86.1+ KB


#### Findings
- There are missing values in Bedrooms, Bathrooms, Garage Available, Furnishing and Has_Pool columns
- Some columns have inconsistent naming like; BedRooms, Bath rooms, year built

In [23]:
# 3)clean column names
print("Original column names:\n")# Print a message to label the output before displaying the original column names
print(df.columns)

Original column names:

Index([' BedRooms ', 'Bath rooms', 'Size_sqft', 'location', 'year built',
       'Garage Available', 'Furnishing', 'House Condition', 'Has_Pool',
       'Lot Size', 'Price($)'],
      dtype='object')


In [35]:
# clean column names
df.columns = df.columns.str.strip() # removes leading and trailing spaces
df.columns = df.columns.str.lower() # converts the column names to lowercase
df.columns = df.columns.str.replace(" ","_") # replaces spaces with underscores
df.columns = df.columns.str.replace(r"[^\w\s]", "", regex=True) # removes special characters

# preview cleaned columns
print("Cleaned column names:\n")# "\n" prints the label then moves to the next line before printing the column names 
print(df.columns)

Cleaned column names:

Index(['bedrooms', 'bath_rooms', 'size_sqft', 'location', 'year_built',
       'garage_available', 'furnishing', 'house_condition', 'has_pool',
       'lot_size', 'price'],
      dtype='object')


#### code explanation
- df.columns = df.columns.str.replace(r"[^\w\s]", "", regex=True)
- df.columns: Accesses the column names of the Dataframe "df".
- .str: Treats the column names as strings allowing me to apply string functions
- .replace(...): Replaces parts of the string; in this case column names, using the pattern provided
- r"[^\w\s]": A regular expression matching all characters except word characters and whitespace:
                \w matches the letters, numbers and underscores
                \s matches whitespace(spaces,tabs)
                ^ inside the brackets means"not"
                So this matches special characters like !@#$%^&* and removes them.
- "": This is the replacement string and since its empty it removes the matched characters
- regex=True: This confirms that the pattern is a regular expression

- The whole process of cleaning columns is important as it ensures column names are; clean, consistent and safe to use for modelling or plotting later.


In [36]:
# 4)check for missing values

# show the number of missing values in each column
print("Missing values in each column:\n")
print(df.isnull().sum())

Missing values in each column:

bedrooms             46
bath_rooms           59
size_sqft             0
location              0
year_built            0
garage_available    104
furnishing            0
house_condition       0
has_pool             55
lot_size              0
price                 0
dtype: int64


#### Code explanation
- print(df.isnull().sum())
- df.isnull(): Returns a Dataframe of the same shape with "True" where values are missing
- .sum(): adds up the "True" values giving a count of missing values per column

In [37]:
# 5)filling in the missing values

#bedrooms and bathrooms columns are numerical and filled in with median to reduce effect of outliers
df["bedrooms"]=df["bedrooms"].fillna(df["bedrooms"].median())
df["bath_rooms"]=df["bath_rooms"].fillna(df["bath_rooms"].median())

# garage_available,has_pool and furnishing filled with mode
df["garage_available"]=df["garage_available"].fillna(df["garage_available"].mode()[0])
df["furnishing"]=df["furnishing"].fillna(df["furnishing"].mode()[0])
df["has_pool"]=df["has_pool"].fillna(df["has_pool"].mode()[0])

#### Reasoning for methods chosen
- bedrooms and bath_rooms columns:median was used since it's a numerical column and skewness was being avoided by large outliers
- garage_available column: mode was used as it is a categorical column thus it's best to use the most frequent value
- furnishing column: mode was used as it is a categorical column thus the assumption of majority class is reasonable
- has_pool column: mode was used as its binary or categorical and using the most frequent is adequate

In [38]:
# confirm missing values after filling
print("Missing values after filling:\n")
print(df.isnull().sum())

Missing values after filling:

bedrooms            0
bath_rooms          0
size_sqft           0
location            0
year_built          0
garage_available    0
furnishing          0
house_condition     0
has_pool            0
lot_size            0
price               0
dtype: int64


In [28]:
# 6)standardizing columns
# view unique values in categorical columns

print("Unique values in 'location':", df['location'].unique()) #the .unique() function shows the distinct values in the location column
print("\nUnique values in 'furnishing':", df['furnishing'].unique())
print("\nUnique values in 'house_condition':", df['house_condition'].unique())


Unique values in 'location': ['Rural' 'Suburban' 'Urban' 'ruraal' 'sub-urban' 'urbn']

Unique values in 'furnishing': ['Unfurnished' '' 'furnised' 'Semi furnished' 'Semi-Furnished' 'Furnished']

Unique values in 'house_condition': ['new' 'Old' 'gd' 'old' 'New' 'Good']


In [31]:
# fix typos 
df["furnishing"].replace({"furnised": "Furnished","Semi furnished": "Semi-Furnished"}, inplace=True)
df["location"].replace({"ruraal": "Rural", "sub-urban": "Suburban", "urbn": "Urban"}, inplace=True)
df["house_condition"].replace({"new": "New", "gd":"Good", "old":"Old"}, inplace=True)
# .replace() function is used to replace typos with the new values in the column

In [12]:
# recheck after cleaning
print("Standardized 'furnishing' values:", df["furnishing"].unique())
print("\nStandardized 'location' values:", df["location"].unique())
print("\nStandardized 'house_condition' values:", df["house_condition"].unique())

Standardized 'furnishing' values: ['Unfurnished' '' 'Furnished' 'Semi-Furnished']

Standardized 'location' values: ['Rural' 'Suburban' 'Urban']

Standardized 'house_condition' values: ['New' 'Old' 'Good']


#### Feature Engineering and Modelling

In [39]:
# 1)derive "age_of_house"=current year - year built
current_year = datetime.now().year
df["age_of_house"] = current_year - df["year_built"]

In [40]:
# derive "price_per_sqft"= price efficiency per square foot
df["price_per_sqft"] = df["price"]/df["size_sqft"]

In [47]:
# rename has_pool column to has_pool_flag and garage_available to has_garage
df.rename(columns={
    "has_pool": "has_pool_flag",
    "garage_available": "has_garage"
}, inplace=True)

In [50]:
# preview the new dataframe
df[["year_built", "age_of_house", "size_sqft", "price", "price_per_sqft", "has_garage", "has_pool_flag"]].head(20)


,year_built,age_of_house,size_sqft,price,price_per_sqft,has_garage,has_pool_flag,has_pool_flag
0,2004,21,1149,244043,212.395997,0,0.0,0
1,1989,36,1169,211250,180.710009,0,0.0,0
2,1993,32,1409,257239,182.568488,0,0.0,0
3,2021,4,1647,310316,188.412872,0,0.0,0
4,2009,16,1865,334791,179.512601,0,1.0,0
5,2020,5,1179,228260,193.604750,0,0.0,0
6,2004,21,2097,361672,172.471149,0,0.0,0
7,1993,32,1392,259003,186.065374,0,0.0,0
8,2014,11,1491,263096,176.456070,0,0.0,0
9,2013,12,1201,242759,202.130724,0,0.0,0


#### Explanation for the columns
- age_of_house: older houses might be cheaper and in need of renovation.necessary for price modelling
- price_per_sqft: this column measures the pricing efficiency per area
